In [1]:
import numpy as np
import pandas as pd
import math
from datamodel import Listing, Observation, Order, OrderDepth, ProsperityEncoder, Symbol, Trade, TradingState
from enum import IntEnum
from statistics import NormalDist
import copy
import matplotlib.pyplot as plt 
from collections import deque, defaultdict
from typing import List, Dict, Tuple, Optional, Any, TypeAlias
import itertools
JSON: TypeAlias = dict[str, "JSON"] | list["JSON"] | str | int | float | bool | None

In [2]:
market_data = pd.read_csv(r"C:\Users\Can\OneDrive\Desktop\IMC\R1\round-1-island-data-bottle\prices_round_1_day_0.csv", sep=";", header=0)
trade_history = pd.read_csv(r"C:\Users\Can\OneDrive\Desktop\IMC\R1\round-1-island-data-bottle\trades_round_1_day_0.csv", sep=";", header=0)

product_symbol = "KELP"
product_symbol = "RAINFOREST_RESIN"
product_symbol = "SQUID_INK"

In [3]:
def get_mid_price(state: TradingState, symbol: str, method, min_vol) -> float:
    order_depth = state.order_depths[symbol]
    if not order_depth.buy_orders or not order_depth.sell_orders:
        return 0.0

    best_bid = max(order_depth.buy_orders.keys())
    best_ask = min(order_depth.sell_orders.keys())
    total_ask = sum(-price * qty for price, qty in order_depth.sell_orders.items())
    total_bid = sum(price * qty for price, qty in order_depth.buy_orders.items())
    qty_ask = sum(-qty for qty in order_depth.sell_orders.values())
    qty_bid = sum(order_depth.buy_orders.values())

    filtered_best_ask = min([p for p, q in order_depth.sell_orders.items() if -q >= min_vol], default=best_ask)
    filtered_best_bid = max([p for p, q in order_depth.buy_orders.items() if q >= min_vol], default=best_bid)

    mid = (best_bid + best_ask) / 2
    weighted_avg = (total_ask + total_bid) / (qty_ask + qty_bid)
    filtered_mid = (filtered_best_ask + filtered_best_bid) / 2 if filtered_best_ask and filtered_best_bid else mid

    return {
        'mid_price': mid,
        'weighted_average': weighted_avg,
        'filtered_mid_price': filtered_mid,
        'ensemble': (weighted_avg + filtered_mid) / 2
    }.get(method, mid)


def compute_volatility(history: deque, window_size: int = 10) -> float:
    if len(history) < window_size:
        return 0.0
    prices = list(history)[-window_size:]
    return float(np.std(prices))


def get_moving_average(history: deque, symbol: str, method: str, window_size: int, min_vol: int) -> Optional[float]:
    if len(history) < window_size:
        return None
    values = [
        get_mid_price(state, symbol, method, min_vol)
        for state in list(history)[-window_size:]
    ]
    return sum(values) / window_size


def aggressive_take_orders(symbol, state, true_value, remaining_buy, remaining_sell,
                           take_width, prevent_adverse=False, adverse_volume=0):
    order_depth = state.order_depths[symbol]
    orders = []

    buy_price_take = math.floor(true_value - take_width)
    sell_price_take = math.ceil(true_value + take_width)

    for price, volume in sorted(order_depth.sell_orders.items()):
        if remaining_buy > 0 and price <= buy_price_take and (not prevent_adverse or abs(volume) <= adverse_volume):
            quantity = min(remaining_buy, -volume)
            orders.append(Order(symbol, price, quantity))
            remaining_buy -= quantity

    for price, volume in sorted(order_depth.buy_orders.items(), reverse=True):
        if remaining_sell > 0 and price >= sell_price_take and (not prevent_adverse or volume <= adverse_volume):
            quantity = min(remaining_sell, volume)
            orders.append(Order(symbol, price, -quantity))
            remaining_sell -= quantity

    return orders, remaining_buy, remaining_sell

def clear_position_liquidation(symbol, state, true_value, position, remaining_buy, remaining_sell, liquidate_width):
    orders = []

    order_depth = state.order_depths[symbol]
    buy_price_liquidate = round(true_value - liquidate_width)
    sell_price_liquidate = round(true_value + liquidate_width)

    to_clear = position - remaining_buy + remaining_sell

    if to_clear > 0:
        clear_quantity = sum(
            volume for price, volume in order_depth.buy_orders.items()
            if price >= sell_price_liquidate
        )
        sent = min(to_clear, clear_quantity)
        if sent > 0:
            orders.append(Order(symbol, sell_price_liquidate, -sent))
            remaining_sell -= sent

    elif to_clear < 0:
        clear_quantity = sum(
            -volume for price, volume in order_depth.sell_orders.items()
            if price <= buy_price_liquidate
        )
        sent = min(-to_clear, clear_quantity)
        if sent > 0:
            orders.append(Order(symbol, buy_price_liquidate, sent))
            remaining_buy -= sent

    return orders, remaining_buy, remaining_sell

def no_liquidation(symbol, state, true_value, position, remaining_buy, remaining_sell, liquidate_width):
    return [], remaining_buy, remaining_sell

def default_market_make(symbol, state, true_value, remaining_buy, remaining_sell,
                        market_make_spread, price_filter_width, fallback_offset):
    orders = []
    order_depth = state.order_depths[symbol]
    sell_prices = [price for price in order_depth.sell_orders if price > true_value + price_filter_width]
    buy_prices = [price for price in order_depth.buy_orders if price < true_value - price_filter_width]
    sell_base = min(sell_prices) if sell_prices else true_value + fallback_offset
    buy_base = max(buy_prices) if buy_prices else true_value - fallback_offset
    buy_price_make = round(buy_base - market_make_spread)
    sell_price_make = round(sell_base + market_make_spread)
    if remaining_buy > 0:
        orders.append(Order(symbol, buy_price_make, remaining_buy))
    if remaining_sell > 0:
        orders.append(Order(symbol, sell_price_make, -remaining_sell))
    return orders

class MarketMakingStrategy:
    def __init__(self, symbol: str, limit: int, print_log: int = 0,
                 take_order_fn=None, take_order_args=None,
                 liquidation_fn=None, liquidation_args=None,
                 market_make_fn=None, market_make_args=None,
                 true_value_args=None):
        self.symbol = symbol
        self.limit = limit
        self.print_log = print_log
        self.price_history_dict = defaultdict(lambda: deque(maxlen=10))
        self.take_order_fn = take_order_fn
        self.take_order_args = take_order_args or {}

        self.liquidation_fn = liquidation_fn
        self.liquidation_args = liquidation_args or {}

        self.market_make_fn = market_make_fn
        self.market_make_args = market_make_args or {}

        self.true_value_args = true_value_args or {}

        # self.history = deque(maxlen=10)
        self.true_value_history = deque(maxlen=30)
        self.window = deque()
        self.window_size = 3

        self.remaining_buy = 0
        self.remaining_sell = 0

        self.orders = []
        self.conversions = 0

    def run(self, state: TradingState) -> tuple[list[Order], int]:
        self.orders = []
        self.conversions = 0
        # self.history.append(copy.deepcopy(state))
        true_value = self.get_true_value(state)
        self.act(state, true_value)
        return self.orders, self.conversions

    def act(self, state: TradingState, true_value: float):
        position = state.position.get(self.symbol, 0)
        self.remaining_buy = self.limit - position
        self.remaining_sell = self.limit + position

        self.window.append(abs(position) == self.limit)
        if len(self.window) > self.window_size:
            self.window.popleft()

        print('startstart') if self.print_log == 1 else None
        print(f"to_buy{self.remaining_buy}...") if self.print_log == 1 else None
        print(f"to_sell{self.remaining_sell}...") if self.print_log == 1 else None
        print(f"position{position}...") if self.print_log == 1 else None

        if self.take_order_fn:
            orders, self.remaining_buy, self.remaining_sell = self.take_order_fn(
                self.symbol, state, true_value,
                self.remaining_buy, self.remaining_sell,
                **self.take_order_args
            )
            self.orders.extend(orders)
            if self.print_log:
                for o in orders:
                    print(f"[{self.symbol}] TAKE → {'BUY' if o.quantity > 0 else 'SELL'} {abs(o.quantity)} @ {o.price}") if self.print_log == 1 else None

        if self.liquidation_fn:
            orders, self.remaining_buy, self.remaining_sell = self.liquidation_fn(
                self.symbol, state, true_value,
                position,
                self.remaining_buy, self.remaining_sell,
                **self.liquidation_args
            )
            self.orders.extend(orders)
            if self.print_log:
                for o in orders:
                    print(f"[{self.symbol}] LIQUIDATE → {'BUY' if o.quantity > 0 else 'SELL'} {abs(o.quantity)} @ {o.price}") if self.print_log == 1 else None

        if self.market_make_fn:
            orders = self.market_make_fn(
                self.symbol, state, true_value,
                self.remaining_buy, self.remaining_sell,
                **self.market_make_args
            )
            self.orders.extend(orders)
            if self.print_log:
                for o in orders:
                    print(f"[{self.symbol}] MAKE → {'BUY' if o.quantity > 0 else 'SELL'} {abs(o.quantity)} @ {o.price}") if self.print_log == 1 else None

        print('endend') if self.print_log == 1 else None

    def get_true_value(self, state: TradingState) -> float:
        use_fixed = self.true_value_args.get("use_fixed", False)
        fixed_value = self.true_value_args.get("fixed_value", None)
        if use_fixed and fixed_value is not None:
            return fixed_value

        method = self.true_value_args.get("method")
        min_vol = self.true_value_args.get("min_vol")
        ma_window = self.true_value_args.get("ma_window")

        current_price = get_mid_price(state, self.symbol, method=method, min_vol=min_vol)

        if self.symbol == "SQUID_INK":
            history = self.price_history_dict[self.symbol]
            history.append(current_price)
            if len(history) >= ma_window:
                return sum(history) / len(history)
            else:
                return current_price
        else:
            return current_price

    def save(self):
        return list(self.window)

    def load(self, data):
        self.window = deque(data) if data else deque()

class Trader:
    def __init__(self):
        self.print_log = 0
        self.symbols = [product_symbol]

        self.strategy_config = {
            product_symbol: {
                "limit": 50,
                "take_order_args": {
                    "prevent_adverse": True,
                    "adverse_volume": 15,
                    "take_width": 10
                },
                "liquidation_args": {
                    "liquidate_width": 1
                },
                "market_make_args": {
                    "market_make_spread": -1,
                    "price_filter_width": 2,
                    "fallback_offset": 2
                },
                "true_value_args": {
                    "use_fixed": False,
                    "method": "weighted_average",
                    "min_vol": 0,
                    "ma_window": 250
                }
            }
        }

        self.cashs = defaultdict(float)
        self.pnl = defaultdict(float)
        self.pnl_history = defaultdict(lambda: deque(maxlen=20))
        self.last_timestamp = -100

        self.recent_prices = {sym: deque(maxlen=10) for sym in self.symbols}
        self.volatility_threshold = {product_symbol: 100}

    def run(self, state: TradingState) -> Tuple[Dict[str, List[Order]], int, str]:
        all_orders = {}
        conversions = 0

        for symbol in self.symbols:
            for trade in state.own_trades.get(symbol, []):
                if trade.timestamp == self.last_timestamp:
                    if trade.buyer == "SUBMISSION":
                        self.cashs[symbol] -= trade.price * trade.quantity
                    elif trade.seller == "SUBMISSION":
                        self.cashs[symbol] += trade.price * trade.quantity

            cfg = self.strategy_config[symbol]

            if cfg["true_value_args"].get("use_fixed", False):
                tv = cfg["true_value_args"]["fixed_value"]
            else:
                tv = get_mid_price(state, symbol, method="weighted_average", min_vol=15)

            price_history = self.recent_prices[symbol]
            price_history.append(tv)

            if len(price_history) == price_history.maxlen:
                returns = np.diff(price_history)
                volatility = np.std(returns)
                if volatility > self.volatility_threshold[symbol]:
                    print(f"[{symbol}] 🚨 High volatility detected! Skipping round.")
                    continue

            strat = MarketMakingStrategy(
                symbol=symbol,
                limit=cfg["limit"],
                print_log=self.print_log,
                take_order_fn=aggressive_take_orders,
                take_order_args=cfg["take_order_args"],
                liquidation_fn=clear_position_liquidation,
                liquidation_args=cfg["liquidation_args"],
                market_make_fn=default_market_make,
                market_make_args=cfg["market_make_args"],
                true_value_args=cfg["true_value_args"]
            )

            orders, conv = strat.run(state)
            tv = strat.get_true_value(state)
            pos = state.position.get(symbol, 0)
            pnl_now = self.cashs[symbol] + pos * tv
            self.pnl[symbol] = pnl_now
            self.pnl_history[symbol].append(pnl_now)

            if self.print_log:
                print(f"[{symbol}] PnL: {pnl_now:.2f}, Cash: {self.cashs[symbol]:.2f}, Pos: {pos}, TV: {tv:.2f}")

            all_orders[symbol] = orders
            conversions += conv

        self.last_timestamp = state.timestamp
        return all_orders, conversions, ""

In [4]:
def _construct_order_depths(group):
    order_depths = {}
    for idx, row in group.iterrows():
        product = row['product']
        order_depth = OrderDepth()
        for i in range(1, 4):
            if f'bid_price_{i}' in row and f'bid_volume_{i}' in row:
                bid_price = row[f'bid_price_{i}']
                bid_volume = row[f'bid_volume_{i}']
                if not pd.isna(bid_price) and not pd.isna(bid_volume):
                    order_depth.buy_orders[int(bid_price)] = int(bid_volume)
            if f'ask_price_{i}' in row and f'ask_volume_{i}' in row:
                ask_price = row[f'ask_price_{i}']
                ask_volume = row[f'ask_volume_{i}']
                if not pd.isna(ask_price) and not pd.isna(ask_volume):
                    order_depth.sell_orders[int(ask_price)] = -int(ask_volume)
        order_depths[product] = order_depth
    return order_depths


def _construct_trading_state(traderData, timestamp, listings, order_depths, 
                                own_trades, market_trades, position, observations):
    state = TradingState(traderData, timestamp, listings, order_depths, own_trades, market_trades, position, observations)
    return state

def _execute_buy_order(timestamp, order, order_depths, position, cash, trade_history_dict, sandboxLog):
        trades = []
        order_depth = order_depths[order.symbol]

        for price, volume in list(order_depth.sell_orders.items()):
            if price > order.price or order.quantity == 0:
                break

            trade_volume = min(abs(order.quantity), abs(volume))
            if abs(trade_volume + position[order.symbol]) <= int(position_limit[order.symbol]):
                trades.append(Trade(order.symbol, price, trade_volume, "SUBMISSION", "", timestamp))
                position[order.symbol] += trade_volume
                cash[order.symbol] = cash[order.symbol] - price * trade_volume
                order_depth.sell_orders[price] += trade_volume
                order.quantity -= trade_volume
            else:
                sandboxLog += f"\nOrders for product {order.symbol} exceeded limit of {position_limit[order.symbol]} set"
            
            if order_depth.sell_orders[price] == 0:
                del order_depth.sell_orders[price]
        
        trades_at_timestamp = trade_history_dict.get(timestamp, [])
        new_trades_at_timestamp = []
        for trade in trades_at_timestamp:
            if trade.symbol == order.symbol:
                if trade.price < order.price:
                    trade_volume = min(abs(order.quantity), abs(trade.quantity))
                    trades.append(Trade(order.symbol, order.price, trade_volume, "SUBMISSION", "", timestamp))
                    order.quantity -= trade_volume
                    position[order.symbol] += trade_volume
                    cash[order.symbol] = cash[order.symbol] - order.price * trade_volume
                    if trade_volume == abs(trade.quantity):
                        continue
                    else:
                        new_quantity = trade.quantity - trade_volume
                        new_trades_at_timestamp.append(Trade(order.symbol, order.price, new_quantity, "", "", timestamp))
                        continue
            new_trades_at_timestamp.append(trade)  

        if len(new_trades_at_timestamp) > 0:
            trade_history_dict[timestamp] = new_trades_at_timestamp

        return trades, sandboxLog
        
def _execute_sell_order(timestamp, order, order_depths, position, cash, trade_history_dict, sandboxLog):
    trades = []
    order_depth = order_depths[order.symbol]
    
    for price, volume in sorted(order_depth.buy_orders.items(), reverse=True):
        if price < order.price or order.quantity == 0:
            break

        trade_volume = min(abs(order.quantity), abs(volume))
        if abs(position[order.symbol] - trade_volume) <= int(position_limit[order.symbol]):
            trades.append(Trade(order.symbol, price, trade_volume, "", "SUBMISSION", timestamp))
            position[order.symbol] -= trade_volume
            cash[order.symbol] = cash[order.symbol] + price * abs(trade_volume)
            order_depth.buy_orders[price] -= abs(trade_volume)
            order.quantity += trade_volume
        else:
            sandboxLog += f"\nOrders for product {order.symbol} exceeded limit of {position_limit[order.symbol]} set"

        if order_depth.buy_orders[price] == 0:
            del order_depth.buy_orders[price]

    trades_at_timestamp = trade_history_dict.get(timestamp, [])
    new_trades_at_timestamp = []
    for trade in trades_at_timestamp:
        if trade.symbol == order.symbol:
            if trade.price > order.price:
                trade_volume = min(abs(order.quantity), abs(trade.quantity))
                trades.append(Trade(order.symbol, order.price, trade_volume, "", "SUBMISSION", timestamp))
                order.quantity += trade_volume
                position[order.symbol] -= trade_volume
                cash[order.symbol] = cash[order.symbol] + order.price * trade_volume
                if trade_volume == abs(trade.quantity):
                    continue
                else:
                    new_quantity = trade.quantity - trade_volume
                    new_trades_at_timestamp.append(Trade(order.symbol, order.price, new_quantity, "", "", timestamp))
                    continue
        new_trades_at_timestamp.append(trade)  

    if len(new_trades_at_timestamp) > 0:
        trade_history_dict[timestamp] = new_trades_at_timestamp
            
    return trades, sandboxLog

def _execute_order(timestamp, order, order_depths, position, cash, trades_at_timestamp, sandboxLog):
    if order.quantity == 0:
        return [],[]
    
    order_depth = order_depths[order.symbol]
    if order.quantity > 0:
        return _execute_buy_order(timestamp, order, order_depths, position, cash, trades_at_timestamp, sandboxLog)
    else:
        return _execute_sell_order(timestamp, order, order_depths, position, cash, trades_at_timestamp, sandboxLog)

def _mark_pnl(cash, position, order_depths, pnl, product):
    order_depth = order_depths[product]
    
    best_ask = min(order_depth.sell_orders.keys())
    best_bid = max(order_depth.buy_orders.keys())
    mid = (best_ask + best_bid)/2
    fair = mid
    
    if product == 'AMETHYSTS':
        fair = 10000
    pnl[product] = cash[product] + fair * position[product]

In [5]:
listings = {
    'RAINFOREST_RESIN': Listing(symbol='RAINFOREST_RESIN', product='RAINFOREST_RESIN', denomination='SEASHELLS'),
    'KELP': Listing(symbol='KELP', product='KELP', denomination='SEASHELLS'),
    'SQUID_INK': Listing(symbol='SQUID_INK', product='SQUID_INK', denomination='SEASHELLS')
}

position_limit = {
    'RAINFOREST_RESIN': 50,
    'KELP': 50,
    'SQUID_INK': 50,
}

timestamp_group_md = market_data.groupby('timestamp')
timestamp_group_th = trade_history.groupby('timestamp')

own_trades = defaultdict(list)
market_trades = defaultdict(list)
pnl_product = defaultdict(float)

trade_history_dict = {}

observations = [Observation({}, {}) for _ in range(len(market_data))]
current_position = {product: 0 for product in listings.keys()}

traderData = "" 

trade_history = trade_history.sort_values(by=['timestamp', 'symbol'])
observations = [Observation({}, {}) for _ in range(len(market_data))]
current_position = {product: 0 for product in listings.keys()}
pnl_history = {'RAINFOREST_RESIN': [],'KELP': [],'SQUID_INK': []}

pnl = {product: 0 for product in listings.keys()}
cash = {product: 0 for product in listings.keys()}
position_history = {'RAINFOREST_RESIN': [],'KELP': [],'SQUID_INK': []}

trades = []
sandbox_logs = []

trade_history_dict = {
    timestamp: [
        Trade(
            row['symbol'],
            int(row['price']),
            int(row['quantity']),
            row['buyer'] if row['buyer'] == 'SUBMISSION' else "",
            row['seller'] if row['seller'] == 'SUBMISSION' else "",
            timestamp
        )
        for _, row in group.iterrows()
    ]
    for timestamp, group in timestamp_group_th
}

In [6]:
def run_backtest_with_config(config: dict) -> float:
    trader = Trader()

    trader.strategy_config[product_symbol]["take_order_args"]["take_width"] = config["take_width"]
    trader.strategy_config[product_symbol]["market_make_args"]["market_make_spread"] = config["market_make_spread"]
    trader.strategy_config[product_symbol]["market_make_args"]["price_filter_width"] = config["price_filter_width"]
    trader.strategy_config[product_symbol]["market_make_args"]["fallback_offset"] = config["fallback_offset"]
    trader.strategy_config[product_symbol]["liquidation_args"]["liquidate_width"] = config["liquidate_width"]
    trader.strategy_config[product_symbol]["true_value_args"]["ma_window"] = config["ma_window"]

    current_position = {product: 0 for product in listings.keys()}
    cash = {product: 0 for product in listings.keys()}
    own_trades = defaultdict(list)
    market_trades = defaultdict(list)
    traderData = ""

    for timestamp, group in timestamp_group_md:
        order_depths = _construct_order_depths(group)
        order_depths_matching = copy.deepcopy(order_depths)

        state = _construct_trading_state(
            traderData, timestamp, listings, order_depths,
            dict(own_trades), dict(market_trades),
            current_position, observations
        )

        orders, conversions, traderData = trader.run(state)
        products = group['product'].tolist()
        sandboxLog = ""
        trades_at_timestamp = trade_history_dict.get(timestamp, [])

        for product in products:
            new_trades = []
            for order in orders.get(product, []):
                trades_done, sandboxLog = _execute_order(
                    timestamp, order, order_depths_matching,
                    current_position, cash, trade_history_dict, sandboxLog
                )
                new_trades.extend(trades_done)
            if new_trades:
                own_trades[product] = new_trades

        if trades_at_timestamp:
            for trade in trades_at_timestamp:
                market_trades[trade.symbol].append(trade)
        else:
            for product in products:
                market_trades[product] = []

    return cash[product_symbol] + current_position[product_symbol] * get_mid_price(state, product_symbol, method="weighted_average", min_vol=15)

In [7]:
# param_grid = {
#     "take_width": [0.5,1,1.5,2],
#     "market_make_spread": [-1,-0.5,0,0.5,1],
#     "price_filter_width": [1,1.5,2],
#     "fallback_offset": [1,1.5,2],
#     "liquidate_width": [0.5,1,1.5,2],
#     "ma_window": [10, 20, 50]
# }
param_grid = {
    "take_width": [1],
    "market_make_spread": [-1],
    "price_filter_width": [2],
    "fallback_offset": [2],
    "liquidate_width": [2],
    "ma_window": [20]
}
keys, values = zip(*param_grid.items())
param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

results = []
for i, config in enumerate(param_combinations):
    print(f"▶ Running {i+1}/{len(param_combinations)}: {config}")
    pnl = run_backtest_with_config(config)
    results.append({"params": config, "pnl": pnl})

# 打印 Top 5
sorted_results = sorted(results, key=lambda x: x["pnl"], reverse=True)
print("\n🏆 Top 5 Parameter Sets:")
for res in sorted_results[:50]:
    print(res)

▶ Running 1/1: {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 2, 'fallback_offset': 2, 'liquidate_width': 2, 'ma_window': 20}

🏆 Top 5 Parameter Sets:
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 2, 'fallback_offset': 2, 'liquidate_width': 2, 'ma_window': 20}, 'pnl': -4442.0}


In [8]:
# SQUID_INK

{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 0.5, 'ma_window': 10}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 10}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 637.5}
{'params': {'take_width': 1.5, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 10}, 'pnl': 590.5}
{'params': {'take_width': 1.5, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 590.5}
{'params': {'take_width': 1.5, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 590.5}
{'params': {'take_width': 1.5, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 0.5, 'ma_window': 10}, 'pnl': 590.5}
{'params': {'take_width': 1.5, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 590.5}
{'params': {'take_width': 1.5, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 590.5}
{'params': {'take_width': 1.5, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 10}, 'pnl': 590.5}
{'params': {'take_width': 1.5, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 590.5}
{'params': {'take_width': 1.5, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 590.5}
{'params': {'take_width': 2, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 10}, 'pnl': 590.5}
{'params': {'take_width': 2, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 590.5}
{'params': {'take_width': 2, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 590.5}
{'params': {'take_width': 2, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 0.5, 'ma_window': 10}, 'pnl': 590.5}
{'params': {'take_width': 2, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 590.5}
{'params': {'take_width': 2, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 590.5}
{'params': {'take_width': 2, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 10}, 'pnl': 590.5}
{'params': {'take_width': 2, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 590.5}
{'params': {'take_width': 2, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 590.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 10}, 'pnl': 579.5}
{'params': {'take_width': 1, 'market_make_spread': 0, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 1.5, 'ma_window': 10}, 'pnl': 434.0}
{'params': {'take_width': 1, 'market_make_spread': 0, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 1.5, 'ma_window': 20}, 'pnl': 434.0}
{'params': {'take_width': 1, 'market_make_spread': 0, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 1.5, 'ma_window': 50}, 'pnl': 434.0}
{'params': {'take_width': 1, 'market_make_spread': 0, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 2, 'ma_window': 10}, 'pnl': 434.0}
{'params': {'take_width': 1, 'market_make_spread': 0, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 2, 'ma_window': 20}, 'pnl': 434.0}
{'params': {'take_width': 1, 'market_make_spread': 0, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 2, 'ma_window': 50}, 'pnl': 434.0}
{'params': {'take_width': 1, 'market_make_spread': 0, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 1.5, 'ma_window': 10}, 'pnl': 434.0}


{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 0.5, 'ma_window': 10}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1.5, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 10}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 637.5}
{'params': {'take_width': 1, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 637.5}
{'params': {'take_width': 1.5, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 10}, 'pnl': 590.5}
{'params': {'take_width': 1.5, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 590.5}
{'params': {'take_width': 1.5, 'market_make_spread': -0.5, 'price_filter_width': 1, 'fallback_offset': 1, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 590.5}

{'params': {'take_width': 1.5,
  'market_make_spread': -0.5,
  'price_filter_width': 1,
  'fallback_offset': 1,
  'liquidate_width': 0.5,
  'ma_window': 50},
 'pnl': 590.5}

In [9]:
# KELP

{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 1, 'ma_window': 20}, 'pnl': 4044.5}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 4027.5}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 3, 'liquidate_width': 1, 'ma_window': 20}, 'pnl': 1565.5}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 3, 'liquidate_width': 1.5, 'ma_window': 20}, 'pnl': 1565.5}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 3, 'ma_window': 20}, 'pnl': 1583.5}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 2, 'ma_window': 20}, 'pnl': 1578.5}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 5, 'liquidate_width': 1, 'ma_window': 20}, 'pnl': 1571.5}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 5, 'liquidate_width': 2, 'ma_window': 20}, 'pnl': 1554.5}

{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 4929.7058823529405}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 1.5, 'ma_window': 20}, 'pnl': 2082.338235294119}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 1, 'ma_window': 20}, 'pnl': 2079.5147058823495}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 2, 'ma_window': 20}, 'pnl': 2071.073529411766}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 3, 'liquidate_width': 1, 'ma_window': 20}, 'pnl': 2064.5294117647063}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 3, 'liquidate_width': 1.5, 'ma_window': 20}, 'pnl': 2064.5294117647063}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 3, 'liquidate_width': 2, 'ma_window': 20}, 'pnl': 2064.5294117647063}

{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 5225.588235294126}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 1, 'ma_window': 20}, 'pnl': 2806.5294117647136}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 5, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 2485.705882352937}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 1.5, 'ma_window': 20}, 'pnl': 2482.5294117647136}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 3, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 2478.0588235294126}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 4, 'liquidate_width': 0.5, 'ma_window': 20}, 'pnl': 2477.705882352937}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 2, 'ma_window': 20}, 'pnl': 2463.5294117647136}

{'params': {'take_width': 1,
  'market_make_spread': -1,
  'price_filter_width': 1,
  'fallback_offset': 2,
  'liquidate_width': 2,
  'ma_window': 20},
 'pnl': 2463.5294117647136}

In [10]:
# RAINFOREST_RESIN

{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 15162.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 1, 'ma_window': 50}, 'pnl': 11298.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 3, 'liquidate_width': 1, 'ma_window': 50}, 'pnl': 11019.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 4, 'liquidate_width': 1, 'ma_window': 50}, 'pnl': 11006.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 5, 'liquidate_width': 1, 'ma_window': 50}, 'pnl': 11003.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 3, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 10992.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 4, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 10967.0}

{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 15651.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 1, 'ma_window': 50}, 'pnl': 11363.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 3, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 11149.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 3, 'liquidate_width': 1, 'ma_window': 50}, 'pnl': 11121.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 4, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 11094.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 5, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 11083.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 4, 'liquidate_width': 1, 'ma_window': 50}, 'pnl': 11082.0}

{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 14633.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 2, 'liquidate_width': 1, 'ma_window': 50}, 'pnl': 11044.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 3, 'liquidate_width': 1, 'ma_window': 50}, 'pnl': 10912.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 4, 'liquidate_width': 1, 'ma_window': 50}, 'pnl': 10877.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 5, 'liquidate_width': 1, 'ma_window': 50}, 'pnl': 10870.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 3, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 10857.0}
{'params': {'take_width': 1, 'market_make_spread': -1, 'price_filter_width': 1, 'fallback_offset': 4, 'liquidate_width': 0.5, 'ma_window': 50}, 'pnl': 10819.0}

{'params': {'take_width': 1,
  'market_make_spread': -1,
  'price_filter_width': 1,
  'fallback_offset': 4,
  'liquidate_width': 0.5,
  'ma_window': 50},
 'pnl': 10819.0}